# 03 — Direct Preference Optimization (DPO) on CPU

**DPO** learns from *preferences* instead of target answers: for each prompt, a preferred ("chosen") answer and a
worse ("rejected") one. It raises the likelihood of the chosen answer relative to the rejected one, compared to a
reference model. With a LoRA adapter the reference is simply the base model with the adapter switched off, so no
second model is loaded.

Same booking task as notebook 01. The chosen answer is the exact JSON; the rejected one is the kind of answer the
base model gives (JSON in a markdown code block, "sunday", "5:15pm") or a subtly wrong value.

Two jobs run in parallel to show a classic pitfall:

| job | loss | what to expect |
|---|---|---|
| `dpo` | DPO only (`sigmoid`) | the gap between chosen and rejected grows, but the model can widen it by moving away from *both*, e.g. by dropping JSON altogether |
| `dpo-sft` | DPO + an SFT term on the chosen answer (`sigmoid+sft`) | keeps the chosen answer likely while learning the preference |

Each job measures exact JSON match and *preference accuracy* (how often the model scores the chosen answer above
the rejected one) before and after 40 steps. Expected: ~9 minutes (both jobs in parallel).

In [ ]:
%pip install -q "sagemaker==3.21.0"

## Setup

`Lab` (in `sm_helpers.py`) holds the steps notebook 01 spells out: the SageMaker session, execution role and S3
bucket, MLflow, and a CPU training job. Two settings:

* `CAPACITY`: how the job gets its instance: `"fallback"` (the first of several CPU types with capacity; the
  default), `"on-demand"` (exactly `ml.c7i.4xlarge`) or `"spot"` (`ml.m5.4xlarge`, cheaper, may wait for capacity). Notebook 01
  compares the three.
* `MLFLOW_APP_ARN`: where the job logs its metrics (see *Track in MLflow* below).

In [ ]:
import sys

sys.path.insert(0, "scripts")
from sm_helpers import Lab

# MLflow: "" = reuse the account's MLflow App (or create one: several minutes, first time only);
# "arn:aws:sagemaker:..." = use that app; None = off. Jobs run the same either way.
MLFLOW_APP_ARN = ""
lab = Lab(mlflow_app_arn=MLFLOW_APP_ARN)
CAPACITY = "fallback"   # "on-demand" | "spot" | "fallback" (compared side by side in 01-sft.ipynb)

In [ ]:
import booking_data

train, _ = booking_data.make_splits()
pair = booking_data.dpo_pairs(train[:1])[0]
print(f"request:  {pair['request']}\nchosen:   {pair['chosen']}\nrejected: {pair['rejected']}")

## Train both variants

Two jobs run `scripts/train_dpo.py` in parallel with TRL's `DPOTrainer`, identical except for `loss_type`. Each
scores the base model, trains for 40 steps on 400 chosen/rejected pairs, and scores again. With a LoRA adapter,
the reference model DPO compares against is the base model itself (adapter switched off), so only one model is in
memory. The wait cell prints status changes for both; if the kernel restarts, re-run setup and the wait cell.

In [ ]:
jobs = {
    "dpo": lab.submit("dpo", "train_dpo.py", {"steps": 40, "loss_type": "sigmoid"},
                      capacity=CAPACITY, requirements="requirements-trl.txt"),
    "dpo-sft": lab.submit("dpo-sft", "train_dpo.py", {"steps": 40, "loss_type": "sigmoid+sft"},
                          capacity=CAPACITY, requirements="requirements-trl.txt"),
}

In [ ]:
jobs = globals().get("jobs") or {k: lab.latest(k) for k in ("dpo", "dpo-sft")}   # after a kernel restart
desc = lab.wait(*jobs.values())

## Results

In [ ]:
print(f"{'':16} {'exact JSON':>10} {'prefers chosen':>15}")
for name, job in jobs.items():
    m = lab.metrics(job)
    if name == "dpo":
        print(f"{'base':16} {m['base']['exact_match']:10.0%} {m['base']['preference_accuracy']:15.0%}")
    print(f"{name:16} {m['finetuned']['exact_match']:10.0%} {m['finetuned']['preference_accuracy']:15.0%}")
print()
for name, job in jobs.items():
    ex = lab.metrics(job)["examples"][0]
    print(f"{name:8} {ex['finetuned']!r}")
print(f"{'target':8} {ex['target']}")

### How to read the results

| Row | Model |
|---|---|
| `base` | Qwen3-0.6B before training |
| `dpo` | trained with the DPO loss only (`loss_type="sigmoid"`) |
| `dpo-sft` | trained with the DPO loss + an SFT loss on the chosen answer (`loss_type="sigmoid+sft"`) |

| Column | What it measures |
|---|---|
| exact JSON | what the model **writes**: on 30 held-out requests, the share of generated answers that parse as JSON and equal the target exactly (bare JSON, `"Sunday"`, `"17:15"`) |
| prefers chosen | what the model **ranks** higher: on 30 held-out chosen/rejected pairs, the share where the model gives the chosen answer a higher probability than the rejected one. Nothing is generated; 50% is a coin flip |

**What happens.** The base model prefers the chosen answer about half the time (~50%) and never writes exact
JSON (0%). After training, *both* variants prefer the chosen answer on (nearly) every pair, yet:

* **`dpo`** still scores 0% exact. Its answers are often not JSON at all (e.g. `name: Priya, party_size: 12, ...`).
  The DPO loss only rewards the **gap** between chosen and rejected, measured against the base model. Here every
  chosen and rejected answer is JSON-like, so the easiest way to widen the gap is to make *both* less likely,
  the rejected one faster. The model ends up preferring the right answer while moving away from JSON altogether.
  This is a known DPO failure mode (sometimes called likelihood displacement).
* **`dpo-sft`** reaches ~90% exact. The extra SFT term also rewards making the chosen answer likely in absolute
  terms, so the gap grows by pushing the chosen answer **up**, not both down.

## Track in MLflow

Each job also logs its parameters, the training loss per step, and the base and fine-tuned scores to MLflow,
one run per job in the experiment `cpu-fine-tuning-techniques`. Select runs and **Compare** to put the techniques
side by side. The link below is valid for a few minutes; re-run the cell for a new one.

In [ ]:
# Compare runs in MLflow: experiment "cpu-fine-tuning-techniques", one run per job (named after the job)
print(lab.mlflow_url() or "MLflow is off")

## Takeaway

Preference accuracy alone can look perfect while the model got worse at the task: always look at what it
generates. In practice, preference tuning starts from a model that already produces the right kind of answer (an
SFT model), or adds an SFT term as here. DPO then refines choices between good answers: tone, safety, which of two
valid answers users prefer.